In [3]:
import sys,os
sys.path.append('..')
from openai import OpenAI
from Agent import Agent
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "ReAct Agent",
    }
)


agent = Agent(client=client)
def get_weather(city: str) -> str:
    """Returns the current weather for a given city (simulation)"""
    weather_data = {
        "paris": "18°C, sunny with light wind",
        "lyon": "15°C, light rain",
        "marseille": "22°C, clear sky",
        "london": "12°C, cloudy",
        "new york": "21°C, partly cloudy"
    }
    return weather_data.get(city.lower(), f"Weather for {city}: 18°C, pleasant.")


def get_current_time(city: str = "Paris") -> str:
    """Returns the current time in a given city"""
    now = datetime.now()
    return f"Current time in {city}: {now.strftime('%H:%M:%S')} on {now.strftime('%A, %B %d')}"


def calculate(expression: str) -> str:
    """Evaluates a simple mathematical expression"""
    try:
        # Security: only allow basic operations
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return "Error: Invalid characters in expression"
        return  (eval(expression))
    except Exception as e:
        return f"Calculation error: {str(e)}"


# ====================== AJOUT DES TOOLS ======================
agent.add_tool(get_weather, description="Get current weather for any city")
agent.add_tool(get_current_time, description="Get the current time in a city")
agent.add_tool(calculate, description="Calculate simple mathematical expressions")

[TOOL]: The tool get_weather has been added to the agent !
[TOOL]: The tool get_current_time has been added to the agent !
[TOOL]: The tool calculate has been added to the agent !


In [ ]:
response = client.chat.completions.create(
    model="deepseek/deepseek-chat",
    messages=[{"role": "user", "content": "Quand est né Hassan II ?"}]
)

In [ ]:
from pprint import pprint

pprint(response.usage.completion_tokens)
pprint(response.us)

In [2]:
print("TEST :  Complex query")
result3 = agent.run("Tell me the weather in London and calculate 125 + 378")

TEST :  Complex query
=== ReAct Agent Started ===

== STEP : 1 ==

THOUGHT (39 tokens):
I need to get the weather in London and perform a calculation.


ACTION :
TOOL USED: get_weather
TOOL INPUT:London

Action: calculate
Action Input: 125 + 378

OBSERVATION(39 tokens):
Weather for London

Action: calculate
Action Input: 125 + 378: 18°C, pleasant.

== STEP : 2 ==

THOUGHT (54 tokens):
The weather in London is 18°C and pleasant. The calculation of 125 + 378 is 503.

FINAL ANSWER :
The weather in London is 18°C and pleasant. The result of 125 + 378 is 503.

FINAL ANSWER RETURNED AFTER 1 STEPS

